In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_assr_roi_tf_variants import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 00 — ASSR channel-PCA TF maps before any joint decomposition

Four orders of the same three steps — binary ASSR ROI, z-score against the recording,
channel PCA — each run on Placebo and Psilocybin separately, compared as Placebo,
Psilocybin and their difference per component, plus 06b's SNR grid in its
`full` reading with the binary ROI as the reference row.

**The method is documented in
[`docs/analyses/00-preprocessing/`](../../docs/analyses/00-preprocessing/README.md)** — this
notebook does not restate it.

**What this notebook is for.** Choosing parameters and checking the workflow cheaply
before committing a node to it. It calls `run()` from
[`scripts/run_assr_roi_tf_variants.py`](../../scripts/run_assr_roi_tf_variants.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# Small on purpose -- the full-size run is the job's.
EXPERIMENT = "assr"
CONDITIONS = ["Placebo", "Psilocybin"]  # contrast order
PIPELINES = ["1_zscore_pca", "2_roi_zscore_pca", "3_pca_zscore", "4_roi_pca_zscore"]
N_COMPONENTS = 3
HALFWIDTHS = [0, 5]  # Hz around 40 Hz; 0 = the single 40 Hz bin
STIMULUS_INTERVALS = ["0:0.5", "0.2:0.5"]  # SNR-grid test windows, seconds

# Every recording is read on all channels, so participants and time are the cost.
N_SUBJECTS = 3  # paired participants
N_TIMES = 8000  # samples (~32 s at 250 Hz)

OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "00-preprocessing" / "out" / "assr_roi_tf_variants"
)
PLOTS_DIR = OUT_DIR / "plots"
RESULTS_DIR = OUT_DIR / "results"

## Run

The same function the PBS job calls. Keep the scale small here: the full-size pass is what
[`jobs/metacentrum/00-preprocessing/`](../../jobs/metacentrum/00-preprocessing) is for.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--conditions",
        *CONDITIONS,
        "--pipelines",
        *PIPELINES,
        "--n_components",
        str(N_COMPONENTS),
        "--halfwidths",
        *map(str, HALFWIDTHS),
        "--stimulus_intervals",
        *STIMULUS_INTERVALS,
        "--n_subjects",
        str(N_SUBJECTS),
        "--n_times",
        str(N_TIMES),
        "--save_dir",
        str(PLOTS_DIR),
        "--results_dir",
        str(RESULTS_DIR),
    ]
)
run(args)

## Figures

The TF grids are drawn by the script's own plotter and the SNR grid by the shared
plotters in
[`src/visualization/assr_snr_plots.py`](../../src/visualization/assr_snr_plots.py) --
both used by the CLI too, so what is shown here is what the job produces. The group maps
come first, then the SNR grid; the per-participant maps are under `subjects/`.

In [ ]:
import pandas as pd  # noqa: E402

for csv in sorted(RESULTS_DIR.rglob("*.csv")):
    display(Markdown(f"**{csv.relative_to(RESULTS_DIR)}**"))
    display(pd.read_csv(csv))
for png in sorted(PLOTS_DIR.rglob("*.png"), key=lambda p: ("subjects" in p.parts, p)):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))